In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from typing import List, Dict, Tuple
from scipy import stats
from sklearn.metrics import roc_curve, auc
# Add necessary paths
sys.path.append('.')
sys.path.append('./plotting_notebooks')
MAIN_AGENT = 'claude-3-7-sonnet-20250219'
# MAIN_AGENT = 'claude-3-5-sonnet-20241022'
MONITOR_AGENT = 'claude-3-5-sonnet-20241022'
TRUSTED_AGENT = 'claude-3-5-haiku-20241022'
# TRUSTED_AGENT = 'gpt-4o'
NUM_TASKS_THRESHOLD = 9
# Import the loading functions from the analysis script
from protocol_plotting_utils import (
    load_in_combined_results,
    load_protocol_intervention_transcripts,
    load_protocol_intervention_data,
    prepare_protocol_monitoring_data,
    merge_protocol_monitoring_with_combined_results,
    make_protocol_results_and_data,
    get_common_tasks,
    filter_protocol_results_and_data,
    prepare_both_system_prompt_data,
    get_uninterrupted_usefulness,
    make_useful_data,
    calculate_task_mean_with_se,
    wilson_conf_interval,
)
from plotting_notebooks.plotting_utils import (
    prepare_monitoring_data,
)
from simulate_markov_utils import plot_markov_df, create_markov_df, add_jitter_to_column
from utils.analysis_utils import balance_classes_with_oversampling

In [ ]:
# Load data
print("Loading data...")

# Load combined results
combined_results = load_in_combined_results(
    include_failed_attempts=True, 
    include_failed_main_task=True
)
combined_results = combined_results[combined_results.transcript_type.str.contains('_v1')]
combined_results['transcript_type'] = combined_results['transcript_type'].str.replace('_v1', '')

# Load protocol data
control_protocol_list = [
    'defer_to_trusted', 
    'defer_to_instructed_trusted_review_instructions', 
    'resample',
    'defer_to_instructed_trusted_low_context', 
    'defer_to_instructed_trusted', 
    'defer_to_trusted_with_feedback', 
    'resample_with_feedback'
]
default_step_mapping = {
    "gpt-4o-2024-08-06": 0,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFx2l:ckpt-step-70': 70,
}
FT_MONITOR_MODELS = list(default_step_mapping.keys())
NUM_MONITORS = 2


protocol_results_and_data = make_protocol_results_and_data(control_protocol_list, MAIN_AGENT, TRUSTED_AGENT)

In [ ]:

common_tasks = get_common_tasks(protocol_results_and_data, num_tasks_threshold=NUM_TASKS_THRESHOLD)
print(f"Tasks common to all protocol+threshold pairs: {sorted(common_tasks)}")
print(f"Number of common tasks: {len(common_tasks)}")

In [ ]:
combined_results = combined_results[combined_results.task_name.isin(common_tasks)]
protocol_results_and_data = filter_protocol_results_and_data(protocol_results_and_data, common_tasks, class_balance=60)

In [ ]:
system_prompt_df_1, system_prompt_df_5 = prepare_both_system_prompt_data(combined_results, common_tasks, base_dir='monitor_responses_v1', include_failed_attempts=False, include_failed_main_task=False, main_agent=MAIN_AGENT, monitor_models=FT_MONITOR_MODELS, num_monitors=NUM_MONITORS)

In [ ]:
combined_results = pd.concat([balance_classes_with_oversampling(combined_results[combined_results.transcript_type == x]) for x in combined_results.transcript_type.unique()])
uninterrupted_usefulness = get_uninterrupted_usefulness(combined_results, MAIN_AGENT, TRUSTED_AGENT)

In [ ]:
def get_normalized_data(df, control_protocol: str, suspicion_threshold: float, column_to_use: str, tasks_to_exclude: List[str] = []):
    filtered_df = df[(df.protocol_name == control_protocol) & (df.suspicion_threshold == suspicion_threshold)]
    # Group by task_name and calculate mean, std, and count
    mean_main_task = filtered_df[filtered_df.transcript_type == 'transcripts_no_secondary_task'].groupby('task_name').agg({column_to_use: 'mean'})
    count_main_task = filtered_df[filtered_df.transcript_type == 'transcripts_no_secondary_task'].groupby('task_name').size().reset_index(name='sample_count')

    usefulness_normalization = uninterrupted_usefulness[(uninterrupted_usefulness.main_agent.isin([MAIN_AGENT, TRUSTED_AGENT]))].reset_index().set_index(['task_name', 'main_agent']).unstack(level='main_agent')[column_to_use]
    # Merge data
    joined_df = pd.merge(mean_main_task, usefulness_normalization, on='task_name', how='left')
    joined_df = pd.merge(joined_df, count_main_task, on='task_name', how='left')
    joined_df = joined_df[~joined_df.task_name.isin(tasks_to_exclude)]
    # Filter tasks where there's a meaningful difference between main and trusted agent
    joined_df = joined_df[joined_df[MAIN_AGENT] - joined_df[TRUSTED_AGENT] > 0.05]
    
    # Calculate normalized values
    joined_df[f'normalized_{column_to_use}'] = (joined_df[column_to_use] - joined_df[TRUSTED_AGENT])/(joined_df[MAIN_AGENT] - joined_df[TRUSTED_AGENT])
    
    # Calculate confidence intervals using Wilson score interval for proportions
    joined_df[f'normalized_{column_to_use}_ci'] = joined_df.apply(
        lambda row: wilson_conf_interval(
            p=row[f'normalized_{column_to_use}'], 
            n=row['sample_count']
        ), 
        axis=1
    )
    
    return joined_df


def weighted_mean_with_wilson_ci(values: np.ndarray, ci_half_widths: np.ndarray) -> Tuple[float, float]:
    """
    Calculate weighted mean and combined confidence interval using inverse-variance weighting.
    Specifically designed for Wilson confidence intervals for proportions.
    
    Args:
        values: Array of proportion values
        ci_half_widths: Array of Wilson confidence interval half-widths
        
    Returns:
        Tuple of (weighted mean, combined confidence interval half-width)
    """
    # Make a copy to avoid modifying the original array
    ci_half_widths = ci_half_widths.copy()
    
    # Add small epsilon to avoid division by zero
    ci_half_widths[ci_half_widths < 0.05] = 0.05
    
    # Calculate weights (inverse of variance, approximated as ci_half_width^2)
    weights = 1 / (ci_half_widths ** 2)
    
    # Calculate weighted mean
    weighted_mean = np.sum(values * weights) / np.sum(weights)

    
    # Calculate combined confidence interval
    combined_ci = 1 / np.sqrt(np.sum(weights))
    
    return weighted_mean, combined_ci

column_to_use = 'main_task_evaluation'


useful_data = make_useful_data(protocol_results_and_data, column_to_use, uninterrupted_usefulness, main_agent=MAIN_AGENT, trusted_agent=TRUSTED_AGENT, tasks_to_exclude = [])

plt.figure(figsize=(8, 6))
for control_protocol in protocol_results_and_data.protocol_name.unique():
    filtered_df = useful_data[useful_data.control_protocol == control_protocol].sort_values(by='suspicion_threshold')
    plt.errorbar(
        filtered_df['suspicion_threshold'], 
        filtered_df['mean'],
        yerr=filtered_df['se'],
        fmt='-o',
        capsize=5,
        label=control_protocol.replace('_', ' ').title()
    )

plt.xlabel('Suspicion Threshold', fontsize=12)
plt.ylabel(f'Mean {column_to_use} across tasks', fontsize=12)
plt.title(f'Protocol Performance vs Suspicion Threshold\n{MAIN_AGENT} - {TRUSTED_AGENT}', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
common_tasks

In [ ]:
# for control_protocol in ['resample']:
#     suspicion_thresholds = protocol_results_and_data[protocol_results_and_data.protocol_name == control_protocol].suspicion_threshold.unique()
#     for suspicion_threshold in sorted(suspicion_thresholds):
#         normalized_data = get_normalized_data(protocol_results_and_data, control_protocol, suspicion_threshold, 'main_task_evaluation', tasks_to_exclude = [])
#         print(suspicion_threshold)
#         display(normalized_data[['task_name', 'normalized_main_task_evaluation']].head())
# #%%
# merge_on = ['task_name', 'transcript_type', 'main_agent', 'baseline_trajectory_id']
# combined_results['baseline_trajectory_id'] = combined_results.file_name.str.split('_').str[-1].str.split('.').str[0]
# protocol_results_and_data = pd.merge(protocol_results_and_data, combined_results[merge_on + ['main_task_evaluation', 'main_task_success']], on=merge_on, how='left', suffixes=('', '_original'))
# protocol_results_and_data = protocol_results_and_data.loc[:, ~protocol_results_and_data.columns.duplicated(keep='first')]
# #%%
# protocol_results_and_data[(protocol_results_and_data.protocol_name == 'defer_to_instructed_trusted_review_instructions') & (protocol_results_and_data.suspicion_threshold == 80) & (protocol_results_and_data.transcript_type == 'transcripts_no_secondary_task') & (protocol_results_and_data.task_name == 'charity_allocation')][['main_task_evaluation', 'main_task_success', 'baseline_trajectory_id']].drop_duplicates().sort_values(by='baseline_trajectory_id')
# #%%
# combined_results[(combined_results.task_name == 'charity_allocation') & (combined_results.main_agent == MAIN_AGENT) & (combined_results.transcript_type == 'transcripts_no_secondary_task')][['file_name', 'main_task_evaluation', 'main_task_success']].drop_duplicates().sort_values(by='file_name')

In [ ]:
plt.figure(figsize=(10, 6))
protocol_results_and_data['suspicion_scores'] = protocol_results_and_data.raw_data.apply(lambda x: [y['suspicion_score'] for y in x['control_metadata']])
protocol_results_and_data['intervening_suspicion_scores'] = protocol_results_and_data.apply(lambda row: [y for y in row['suspicion_scores'] if y > row['suspicion_threshold']], axis=1)
for main_task_success_original in [True, False]:
    plt.hist(protocol_results_and_data[(protocol_results_and_data.transcript_type == 'transcripts_no_secondary_task') & (protocol_results_and_data.main_task_success != protocol_results_and_data.main_task_success_original) & (protocol_results_and_data.main_task_success_original == main_task_success_original)]['suspicion_scores'].explode().values, bins=np.linspace(20, 100, 41), alpha=0.5, density=True, label='Got worse' if main_task_success_original else 'Got better')
plt.title(f'All Suspicion Scores for Main Task Success based on original main task failure\n{MAIN_AGENT} - {TRUSTED_AGENT}')
plt.xlabel('Suspicion Score')
plt.ylabel('Density')
plt.legend()
plt.show()


In [ ]:
protocol = 'defer_to_trusted'
for main_task_success_original in [True, False]:
    plt.hist(protocol_results_and_data[(protocol_results_and_data.transcript_type == 'transcripts_no_secondary_task') & (protocol_results_and_data.main_task_success != protocol_results_and_data.main_task_success_original) & (protocol_results_and_data.main_task_success_original == main_task_success_original) & (protocol_results_and_data.protocol_name == protocol)]['intervening_suspicion_scores'].explode().values, bins=np.linspace(40, 100, 41), alpha=0.5, density=True, label='Got worse' if main_task_success_original else 'Got better')
plt.title(f'Intervening Suspicion Scores for Main Task Success based on original main task failure\n{MAIN_AGENT} - {TRUSTED_AGENT}\n{protocol}')
plt.xlabel('Suspicion Score')
plt.ylabel('Density')
plt.legend()
plt.show()